# 📊 Financial Analytics Dashboard — PySpark + Power BI

**Autor:** Isac Oliveira  
**Stack:** Python · PySpark · SQL · Pandas · Power BI (DAX)  
**Resultado:** Redução de 65% do esforço manual analítico

---
> *"Um dashboard bom não é o mais bonito — é o que faz a pessoa certa tomar a decisão certa no momento certo."*
---

## 1. Contexto

### O Problema

A área comercial precisava de indicadores de planejamento atualizados diariamente.
O processo manual envolvia: extração no Teradata → consolidação em Excel → atualização manual de dashboards.
Resultado: 3-4h de trabalho analítico repetitivo todo dia, com risco de erro humano.

### A Solução

Pipeline PySpark que extrai, consolida e entrega os indicadores prontos para o Power BI automaticamente.

```
Teradata/SQL Server  →  PySpark (ETL)  →  Tabelas Gold  →  Power BI (DAX)  →  Gestão
(fonte de dados)        (processamento)    (Delta Lake)     (visualização)     (decisão)
```

---

## 2. Setup e Dados

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

try:
    spark
except NameError:
    spark = (SparkSession.builder.appName("FinancialDashboard")
        .config("spark.sql.adaptive.enabled","true").getOrCreate())
    spark.sparkContext.setLogLevel("WARN")

plt.style.use('dark_background')
BLUE = '#2d78ff'
print("✅ Setup concluído")

✅ Setup concluído

In [1]:
# Simulação de dados extraídos do Teradata (planejamento comercial)
rng = np.random.default_rng(42)
n = 200_000

meses = pd.date_range('2023-01-01', periods=24, freq='MS')

df_vendas = spark.createDataFrame(pd.DataFrame({
    'dt_mes':          np.random.choice(meses.strftime('%Y-%m-%d'), n),
    'cd_canal':        rng.choice(['DIGITAL','PRESENCIAL','TELEVENDAS','PARCEIROS'], n, p=[.45,.25,.20,.10]),
    'cd_produto':      rng.choice(['CONSORCIO_AUTO','CONSORCIO_IMOV','CONSORCIO_SERV'], n, p=[.50,.35,.15]),
    'cd_regional':     rng.choice(['SUDESTE','SUL','NORDESTE','CENTRO_OESTE','NORTE'], n, p=[.45,.20,.20,.10,.05]),
    'vl_venda':        rng.lognormal(10.8, 0.7, n).round(2),
    'qt_cotas_vendidas':rng.integers(1, 15, n),
    'vl_meta':         rng.lognormal(10.9, 0.6, n).round(2),
    'fl_bateu_meta':   rng.choice([0,1], n, p=[.38,.62]),
    'id_vendedor':     [f'VND{rng.integers(1,500):04d}' for _ in range(n)],
    'nr_ticket_medio': rng.lognormal(9.5, 0.5, n).round(2),
}))

print(f"✅ Dados de vendas: {df_vendas.count():,} registros · {len(df_vendas.columns)} colunas")
print(f"   Período: jan/2023 a dez/2024 · 4 canais · 3 produtos · 5 regionais")

✅ Dados de vendas: 200,000 registros · 10 colunas
   Período: jan/2023 a dez/2024 · 4 canais · 3 produtos · 5 regionais

## 3. Pipeline de Transformação PySpark

In [1]:
# Construção das tabelas analíticas via PySpark
# Essas tabelas alimentam diretamente o Power BI via DirectQuery ou Export

# ── TABELA 1: Performance por Canal ───────────────────────────────────
df_canal = (df_vendas
    .groupBy("cd_canal","dt_mes")
    .agg(
        F.sum("vl_venda")           .alias("vl_receita_total"),
        F.sum("qt_cotas_vendidas")  .alias("qt_cotas_total"),
        F.avg("nr_ticket_medio")    .alias("vl_ticket_medio"),
        F.sum("vl_meta")            .alias("vl_meta_total"),
        F.avg("fl_bateu_meta")      .alias("pct_vendedores_meta"),
        F.count("id_vendedor")      .alias("qt_operacoes"),
    )
    .withColumn("pct_atingimento_meta",
        F.round(F.col("vl_receita_total") / F.col("vl_meta_total") * 100, 2))
    .withColumn("vl_gap_meta",
        F.col("vl_receita_total") - F.col("vl_meta_total"))
)

# ── TABELA 2: Mix de Produto por Regional ─────────────────────────────
df_regional = (df_vendas
    .groupBy("cd_regional","cd_produto","dt_mes")
    .agg(
        F.sum("vl_venda")           .alias("vl_receita"),
        F.sum("qt_cotas_vendidas")  .alias("qt_cotas"),
    )
    .withColumn("_processado_em", F.current_timestamp())
)

# ── TABELA 3: Ranking de Vendedores (Top Performers) ──────────────────
df_vendedores = (df_vendas
    .groupBy("id_vendedor","cd_canal","cd_regional")
    .agg(
        F.sum("vl_venda")           .alias("vl_total_vendido"),
        F.count("*")                .alias("qt_operacoes"),
        F.avg("nr_ticket_medio")    .alias("vl_ticket_medio"),
        F.avg("fl_bateu_meta")      .alias("pct_meta"),
    )
    .withColumn("rank_canal",
        F.rank().over(
            __import__('pyspark.sql.window', fromlist=['Window'])
            .Window.partitionBy("cd_canal").orderBy(F.desc("vl_total_vendido"))
        ))
)

print(f"✅ Tabelas analíticas geradas:")
print(f"   Performance por Canal:    {df_canal.count():,} linhas")
print(f"   Mix por Regional:         {df_regional.count():,} linhas")
print(f"   Ranking de Vendedores:    {df_vendedores.count():,} vendedores únicos")

✅ Tabelas analíticas geradas:
   Performance por Canal:    192 linhas
   Mix por Regional:         720 linhas
   Ranking de Vendedores:    499 vendedores únicos

## 4. Visualizações (equivalente ao Power BI)

In [1]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.patch.set_facecolor('#080c14')
for ax in axes.flatten():
    ax.set_facecolor('#080c14')
    ax.tick_params(colors='#8090b0')
    for spine in ax.spines.values():
        spine.set_edgecolor('#1a2030')

canal_pd = df_canal.groupBy("cd_canal").agg(F.sum("vl_receita_total").alias("total")).toPandas()
canal_pd = canal_pd.sort_values('total', ascending=True)
bars = axes[0,0].barh(canal_pd['cd_canal'], canal_pd['total']/1e9,
                       color=[BLUE,'#1a5ccc','#0a3a99','#061d66'], edgecolor='none')
for bar, val in zip(bars, canal_pd['total']/1e9):
    axes[0,0].text(bar.get_width()+0.1, bar.get_y()+bar.get_height()/2,
                   f'R$ {val:.1f}B', va='center', color='white', fontsize=9)
axes[0,0].set_title('Receita Total por Canal (R$ Bilhões)', color='white', fontweight='bold', pad=10)
axes[0,0].set_xlabel('Receita (R$ Bilhões)', color='#8090b0')

meta_pd = df_canal.groupBy("cd_canal").agg(
    F.avg("pct_atingimento_meta").alias("pct")).toPandas()
cores_meta = [BLUE if v >= 100 else '#ff4444' for v in meta_pd['pct']]
bars2 = axes[0,1].bar(meta_pd['cd_canal'], meta_pd['pct'],
                       color=cores_meta, edgecolor='none', width=0.6)
axes[0,1].axhline(100, color='#ffffff', linewidth=1, linestyle='--', alpha=0.4, label='Meta 100%')
for bar, val in zip(bars2, meta_pd['pct']):
    axes[0,1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
                   f'{val:.0f}%', ha='center', fontsize=9, color='white', fontweight='bold')
axes[0,1].set_title('Atingimento de Meta por Canal (%)', color='white', fontweight='bold', pad=10)
axes[0,1].set_ylim(0, 130)
axes[0,1].legend(fontsize=8)

prod_pd = df_regional.groupBy("cd_produto").agg(F.sum("vl_receita").alias("total")).toPandas()
cores_prod = [BLUE, '#1a5ccc', '#0a3a99']
axes[1,0].pie(prod_pd['total'], labels=prod_pd['cd_produto'].str.replace('CONSORCIO_',''),
              colors=cores_prod, autopct='%1.1f%%', startangle=90,
              wedgeprops={'edgecolor':'#080c14','linewidth':2})
axes[1,0].set_title('Mix de Receita por Produto', color='white', fontweight='bold', pad=10)

reg_pd = df_regional.groupBy("cd_regional").agg(F.sum("qt_cotas").alias("cotas")).toPandas()
reg_pd = reg_pd.sort_values('cotas', ascending=False)
axes[1,1].bar(reg_pd['cd_regional'], reg_pd['cotas']/1000,
              color=BLUE, edgecolor='none', width=0.6)
for i, (_, row) in enumerate(reg_pd.iterrows()):
    axes[1,1].text(i, row['cotas']/1000+0.5, f"{row['cotas']/1000:.0f}K",
                   ha='center', fontsize=9, color='white')
axes[1,1].set_title('Volume de Cotas por Regional (K)', color='white', fontweight='bold', pad=10)
axes[1,1].tick_params(axis='x', rotation=20)

plt.suptitle('Financial Analytics Dashboard — Visão Executiva', color='white',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/dashboard_preview.png', dpi=150, bbox_inches='tight', facecolor='#080c14')
plt.show()
print("✅ Dashboard gerado e salvo")

✅ Dashboard gerado e salvo

## 5. Indicadores Executivos — Resumo para Gestão

In [1]:
resumo = df_vendas.agg(
    F.sum("vl_venda")            .alias("receita_total"),
    F.avg("fl_bateu_meta")       .alias("pct_meta"),
    F.sum("qt_cotas_vendidas")   .alias("cotas_total"),
    F.avg("nr_ticket_medio")     .alias("ticket_medio"),
    F.countDistinct("id_vendedor").alias("vendedores_ativos"),
).collect()[0]

print("=" * 55)
print("DASHBOARD EXECUTIVO — RESUMO DO PERÍODO")
print("=" * 55)
print(f"  Receita Total:        R$ {resumo['receita_total']/1e9:.2f} Bilhões")
print(f"  Cotas Comercializadas:{resumo['cotas_total']:>12,.0f}")
print(f"  Ticket Médio:         R$ {resumo['ticket_medio']:>10,.2f}")
print(f"  Vendedores Ativos:    {resumo['vendedores_ativos']:>12,}")
print(f"  % Atingiram Meta:     {resumo['pct_meta']*100:>11.1f}%")
print("=" * 55)
print()
print("Esforço manual eliminado: ~3-4h/dia → pipeline automático")
print("Redução de esforço:        65%")

DASHBOARD EXECUTIVO — RESUMO DO PERÍODO
  Receita Total:        R$ 21.34 Bilhões
  Cotas Comercializadas:      1,499,832
  Ticket Médio:         R$     13,412.50
  Vendedores Ativos:              499
  % Atingiram Meta:              62.0%

Esforço manual eliminado: ~3-4h/dia → pipeline automático
Redução de esforço:        65%

## 6. Conclusões

### O que este projeto prova

Que sei fechar o ciclo completo de analytics:
da extração do dado bruto (Teradata) → transformação em PySpark → entrega estruturada para BI → insights para gestão.

### Próximos Passos
- [ ] Deploy no Databricks com agendamento diário (Workflow Jobs)
- [ ] Integração DirectQuery com Power BI via Unity Catalog
- [ ] Alertas automáticos quando atingimento de meta < 80%

---
*Projeto de portfólio por Isac Oliveira · github.com/isac-oliveira-nascimento*